##**Step 1 : Install dependency packages**

In [ ]:
%pip install google-api-python-client  openai python-dotenv litellm rapidfuzz dateparser

## **Step 2 : Imports packages**

In [ ]:
import os
import json
import pandas as pd
from rapidfuzz import process
import dateparser
from openai import OpenAI
from dotenv import load_dotenv

## **Step 3 : Load API key**

In [ ]:
load_dotenv()

APIKEY = os.getenv("OPENAI_API_KEY")

if not APIKEY:
    raise ValueError("Missing OPENAI_API_KEY")

client = OpenAI(api_key=APIKEY)


## **Step 4 : Load Dataset**

In [ ]:
df = pd.read_csv("listings.csv")

# Feature engineering
df["occupancy_rate"] = (365 - df["availability_365"]) / 365

df.head()

## **Step 5 : Helper Functions**

In [ ]:
def match_neighbourhood(user_input):
    choices = df["neighbourhood"].dropna().unique()
    match = process.extractOne(user_input.lower(), choices)
    return match[0] if match else None


ROOM_TYPE_MAP = {
    "entire": "entire home/apt",
    "apartment": "entire home/apt",
    "private": "private room",
    "shared": "shared room",
    "hotel": "hotel room"
}

def map_room_type(user_input):
    if not user_input:
        return None
    user_input = user_input.lower()
    for key, value in ROOM_TYPE_MAP.items():
        if key in user_input:
            return value
    return None


def normalize_dates(date_text):
    try:
        parts = date_text.split("-")
        start = dateparser.parse(parts[0]).strftime("%Y-%m-%d")
        end = dateparser.parse(parts[1]).strftime("%Y-%m-%d")
        return start, end
    except:
        return None, None

##**Step 6: Tool Functions (Agent Tools)**

In [ ]:
def search_listings(neighbourhood):
    matched = match_neighbourhood(neighbourhood)

    if not matched:
        return {"status": "failed", "message": "No matching area found"}

    results = (
        df[df["neighbourhood"] == matched]
        .sort_values(by="occupancy_rate", ascending=False)
        .head(10)
    )

    return {
        "status": "success",
        "neighbourhood": matched,
        "results": results[
            ["name", "room_type", "price", "occupancy_rate"]
        ].to_dict(orient="records"),
    }


def book_listing(name, room_type, start_date, end_date):
    match = process.extractOne(name, df["name"])
    if not match:
        return {"status": "failed", "message": "Listing not found"}

    matched_name = match[0]
    mapped_room = map_room_type(room_type)

    listing = df[df["name"] == matched_name]

    if mapped_room:
        listing = listing[listing["room_type"] == mapped_room]

    if listing.empty:
        return {"status": "failed", "message": "Room type not available"}

    return {
        "status": "success",
        "message": f"Booked {matched_name} ({mapped_room}) from {start_date} to {end_date}",
    }

##**Step 7 : Run agent**

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_listings",
            "description": "Search Airbnb listings by neighbourhood",
            "parameters": {
                "type": "object",
                "properties": {
                    "neighbourhood": {"type": "string"}
                },
                "required": ["neighbourhood"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "book_listing",
            "description": "Book a listing",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "room_type": {"type": "string"},
                    "start_date": {"type": "string"},
                    "end_date": {"type": "string"},
                },
                "required": ["name", "room_type", "start_date", "end_date"],
            },
        },
    },
]

In [ ]:
def run_agent():
    messages = [
        {
            "role": "system",
            "content": """
You are an Airbnb booking assistant working ONLY with internal dataset.

RULES:
- Always call search_listings before recommending
- Never invent listings
- Use only tool results
- Focus on converting user to booking
"""
        }
    ]

    while True:
        user_input = input("\nUser: ")

        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        messages.append({"role": "user", "content": user_input})

        while True:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=messages,
                tools=tools,
                tool_choice="auto",
            )

            msg = response.choices[0].message

            # Tool call
            if msg.tool_calls:
                messages.append(msg)

                for tool_call in msg.tool_calls:
                    name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)

                    if name == "search_listings":
                        result = search_listings(**args)
                    elif name == "book_listing":
                        result = book_listing(**args)
                    else:
                        result = {"status": "failed", "message": "Unknown tool"}

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(result)
                    })

                continue

            # Final response
            print("\nAssistant:", msg.content)
            messages.append(msg)
            break

In [ ]:
run_agent()